In [2]:
# Лабораторная работа: Минимальное число ходов шахматного коня
# Реализация структуры данных "Очередь" тремя способами

from collections import deque
import time
from typing import List, Tuple, Optional


# АВТОР И ГРУППА

def print_author_info():
    """Вывод информации об авторе"""
    print("Лукьянец Антон Дмитриевич")
    print("090304-РПИа-о25")
    print()


# РЕАЛИЗАЦИЯ А: Очередь на основе массива (списка)

class QueueArray:
    """Очередь, реализованная на основе динамического массива"""

    def __init__(self):
        self._items = []

    def enqueue(self, item):
        """Добавление элемента в конец очереди"""
        self._items.append(item)

    def dequeue(self):
        """Удаление элемента из начала очереди"""
        if self.is_empty():
            raise IndexError(" dequeue from empty queue")
        return self._items.pop(0)

    def is_empty(self):
        """Проверка очереди на пустоту"""
        return len(self._items) == 0

    def size(self):
        """Возврат размера очереди"""
        return len(self._items)

    def peek(self):
        """Просмотр первого элемента без удаления"""
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._items[0]


# РЕАЛИЗАЦИЯ Б: Очередь на основе связного списка

class Node:
    """Узел связного списка"""

    def __init__(self, data):
        self.data = data
        self.next = None


class QueueLinkedList:
    """Очередь, реализованная на основе связного списка"""

    def __init__(self):
        self._front = None  # Начало очереди
        self._rear = None   # Конец очереди
        self._size = 0

    def enqueue(self, item):
        """Добавление элемента в конец очереди"""
        new_node = Node(item)
        if self._rear is None:
            # Очередь пуста
            self._front = self._rear = new_node
        else:
            # Добавляем в конец
            self._rear.next = new_node
            self._rear = new_node
        self._size += 1

    def dequeue(self):
        """Удаление элемента из начала очереди"""
        if self.is_empty():
            raise IndexError("dequeue from empty queue")

        removed_node = self._front
        self._front = self._front.next

        if self._front is None:
            # Очередь стала пустой
            self._rear = None

        self._size -= 1
        return removed_node.data

    def is_empty(self):
        """Проверка очереди на пустоту"""
        return self._front is None

    def size(self):
        """Возврат размера очереди"""
        return self._size

    def peek(self):
        """Просмотр первого элемента без удаления"""
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._front.data


# РЕАЛИЗАЦИЯ В: Очередь на основе стандартной библиотеки Python

class QueueStandard:
    """Очередь на основе collections.deque"""

    def __init__(self):
        self._deque = deque()

    def enqueue(self, item):
        """Добавление элемента в конец очереди"""
        self._deque.append(item)

    def dequeue(self):
        """Удаление элемента из начала очереди"""
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        return self._deque.popleft()

    def is_empty(self):
        """Проверка очереди на пустоту"""
        return len(self._deque) == 0

    def size(self):
        """Возврат размера очереди"""
        return len(self._deque)

    def peek(self):
        """Просмотр первого элемента без удаления"""
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._deque[0]


# РЕШЕНИЕ ЗАДАЧИ: Минимальное число ходов коня

class ChessKnight:
    """Класс для решения задачи о минимальном числе ходов коня"""

    # Возможные ходы коня
    KNIGHT_MOVES = [
        (2, 1), (2, -1), (-2, 1), (-2, -1),
        (1, 2), (1, -2), (-1, 2), (-1, -2)
    ]

    def __init__(self, queue_class, queue_name):
        self.queue_class = queue_class
        self.queue_name = queue_name

    def parse_position(self, pos: str) -> Tuple[int, int]:
        """
        Преобразование шахматной нотации в координаты
        Например: 'A5' -> (0, 4)
        """
        col = ord(pos[0].upper()) - ord('A')  # 0-7
        row = int(pos[1]) - 1                  # 0-7
        return (row, col)

    def position_to_notation(self, row: int, col: int) -> str:
        """
        Преобразование координат в шахматную нотацию
        Например: (0, 4) -> 'E1'
        """
        col_letter = chr(ord('A') + col)
        row_number = row + 1
        return f"{col_letter}{row_number}"

    def is_valid_position(self, row: int, col: int) -> bool:
        """Проверка, находится ли позиция на доске"""
        return 0 <= row < 8 and 0 <= col < 8

    def get_knight_moves(self, row: int, col: int) -> List[Tuple[int, int]]:
        """Получение всех возможных ходов коня из данной позиции"""
        moves = []
        for dr, dc in self.KNIGHT_MOVES:
            new_row, new_col = row + dr, col + dc
            if self.is_valid_position(new_row, new_col):
                moves.append((new_row, new_col))
        return moves

    def find_min_moves(self, start: str, end: str) -> Tuple[int, List[str]]:
        """
        Поиск минимального числа ходов коня из start в end
        Использует BFS (поиск в ширину)

        Возвращает: (число ходов, путь)
        """
        start_pos = self.parse_position(start)
        end_pos = self.parse_position(end)

        # Если начальная и конечная позиции совпадают
        if start_pos == end_pos:
            return 0, [start]

        # BFS
        queue = self.queue_class()
        queue.enqueue((start_pos, [start]))  # (позиция, путь)
        visited = {start_pos}

        while not queue.is_empty():
            (row, col), path = queue.dequeue()

            # Проверяем все возможные ходы
            for new_row, new_col in self.get_knight_moves(row, col):
                if (new_row, new_col) not in visited:
                    new_pos = (new_row, new_col)
                    new_notation = self.position_to_notation(new_row, new_col)
                    new_path = path + [new_notation]

                    # Если достигли цели
                    if new_pos == end_pos:
                        return len(new_path) - 1, new_path

                    visited.add(new_pos)
                    queue.enqueue((new_pos, new_path))

        # Если путь не найден
        return -1, []


# ТЕСТИРОВАНИЕ И СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ

def test_queue_implementations():
    """Тестирование всех реализаций очереди"""
    print("ТЕСТИРОВАНИЕ РЕАЛИЗАЦИЙ ОЧЕРЕДИ")

    test_data = [1, 2, 3, 4, 5]

    implementations = [
        (QueueArray, "Массив"),
        (QueueLinkedList, "Связный список"),
        (QueueStandard, "Стандартная библиотека")
    ]

    for queue_class, name in implementations:
        print(f"\n{name}:")
        queue = queue_class()

        # Тест enqueue
        for item in test_data:
            queue.enqueue(item)
        print(f"  После добавления {test_data}: размер = {queue.size()}")

        # Тест peek
        print(f"  Первый элемент (peek): {queue.peek()}")

        # Тест dequeue
        dequeued = [queue.dequeue() for _ in range(len(test_data))]
        print(f"  Извлечено: {dequeued}")
        print(f"  Очередь пуста: {queue.is_empty()}")

        print(f"  ✓ Тест пройден")


def test_knight_moves():
    """Тестирование решения задачи о ходе коня"""
    print()
    print("ТЕСТИРОВАНИЕ ЗАДАЧИ О ХОДЕ КОНЯ")

    test_cases = [
        ("A1", "B3", 1),
        ("A1", "C2", 1),
        ("A1", "H8", 6),
        ("A5", "C2", 3),
        ("A1", "A1", 0),
        ("A1", "H1", 5),
    ]

    knight = ChessKnight(QueueStandard, "Стандартная")

    for start, end, expected in test_cases:
        moves, path = knight.find_min_moves(start, end)
        status = "✓" if moves == expected else "✗"
        print(f"\n{status} {start} → {end}:")
        print(f"   Минимум ходов: {moves} (ожидалось: {expected})")
        print(f"   Путь: {' → '.join(path)}")


def compare_performance():
    """Сравнение производительности трех реализаций"""
    print()
    print("СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ")

    # Тестовые данные
    test_positions = [
        ("A1", "H8"),
        ("A1", "A1"),
        ("B2", "G7"),
        ("A5", "C2"),
        ("D4", "D4"),
        ("A1", "H1"),
        ("C3", "F6"),
        ("E4", "E4"),
        ("G1", "G8"),
        ("A8", "H1"),
    ]

    implementations = [
        (QueueArray, "Массив"),
        (QueueLinkedList, "Связный список"),
        (QueueStandard, "Стандартная библиотека (deque)")
    ]

    print("\nПоиск минимального числа ходов коня:")
    print(f"{'Позиции':<15} {'Массив':<20} {'Связный список':<20} {'deque':<15}")

    results = {name: [] for _, name in implementations}

    for start, end in test_positions:
        row = f"{start} → {end}"
        times = {}

        for queue_class, name in implementations:
            knight = ChessKnight(queue_class, name)

            # Замер времени
            start_time = time.perf_counter()

            # Выполняем несколько итераций для точности
            for _ in range(100):
                moves, path = knight.find_min_moves(start, end)

            end_time = time.perf_counter()
            avg_time = (end_time - start_time) / 100 * 1000  # в миллисекундах

            times[name] = avg_time
            results[name].append(avg_time)

        print(f"{row:<15} {times['Массив']:<20.4f} {times['Связный список']:<20.4f} {times['Стандартная библиотека (deque)']:<15.4f}")

    # Вывод статистики
    print()
    print("СРЕДНЕЕ ВРЕМЯ (мс):")

    for queue_class, name in implementations:
        avg = sum(results[name]) / len(results[name])
        print(f"{name:<30} {avg:.4f} мс")

    print()
    print("ВЫВОДЫ:")

    fastest = min(results.keys(), key=lambda x: sum(results[x])/len(results[x]))
    slowest = max(results.keys(), key=lambda x: sum(results[x])/len(results[x]))

    print(f"✓ Быстрее всего: {fastest}")
    print(f"✗ Медленнее всего: {slowest}")
    print("\nПричины различий в производительности:")
    print("  - Массив: pop(0) требует сдвига всех элементов O(n)")
    print("  - Связный список: удаление из начала O(1), но больше накладных расходов")
    print("  - deque: оптимизированная двусторонняя очередь O(1) на обе операции")


def main():
    """Основная функция"""
    print_author_info()

    # Тестирование реализаций очереди
    test_queue_implementations()

    # Тестирование задачи о ходе коня
    test_knight_moves()

    # Сравнение производительности
    compare_performance()

    # Интерактивный режим
    print()
    print("ИНТЕРАКТИВНЫЙ РЕЖИМ")
    print("Введите две позиции (например: A5 C2) или 'exit' для выхода")

    knight = ChessKnight(QueueStandard, "Стандартная")

    while True:
        try:
            user_input = input("\nВаш ввод: ").strip().upper()

            if user_input == 'EXIT':
                print("Завершение работы...")
                break

            positions = user_input.split()
            if len(positions) != 2:
                print("Ошибка: введите две позиции через пробел")
                continue

            start, end = positions

            # Проверка корректности позиций
            if not (len(start) == 2 and len(end) == 2 and
                    start[0] in 'ABCDEFGH' and end[0] in 'ABCDEFGH' and
                    start[1] in '12345678' and end[1] in '12345678'):
                print("Ошибка: некорректный формат позиции")
                continue

            moves, path = knight.find_min_moves(start, end)

            print(f"\nРезультат:")
            print(f"  Минимальное число ходов: {moves}")
            print(f"  Путь: {' → '.join(path)}")

        except KeyboardInterrupt:
            print("\nЗавершение работы...")
            break
        except Exception as e:
            print(f"Ошибка: {e}")


if __name__ == "__main__":
    main()

Лукьянец Антон Дмитриевич
090304-РПИа-о25

ТЕСТИРОВАНИЕ РЕАЛИЗАЦИЙ ОЧЕРЕДИ

Массив:
  После добавления [1, 2, 3, 4, 5]: размер = 5
  Первый элемент (peek): 1
  Извлечено: [1, 2, 3, 4, 5]
  Очередь пуста: True
  ✓ Тест пройден

Связный список:
  После добавления [1, 2, 3, 4, 5]: размер = 5
  Первый элемент (peek): 1
  Извлечено: [1, 2, 3, 4, 5]
  Очередь пуста: True
  ✓ Тест пройден

Стандартная библиотека:
  После добавления [1, 2, 3, 4, 5]: размер = 5
  Первый элемент (peek): 1
  Извлечено: [1, 2, 3, 4, 5]
  Очередь пуста: True
  ✓ Тест пройден

ТЕСТИРОВАНИЕ ЗАДАЧИ О ХОДЕ КОНЯ

✓ A1 → B3:
   Минимум ходов: 1 (ожидалось: 1)
   Путь: A1 → B3

✓ A1 → C2:
   Минимум ходов: 1 (ожидалось: 1)
   Путь: A1 → C2

✓ A1 → H8:
   Минимум ходов: 6 (ожидалось: 6)
   Путь: A1 → B3 → C5 → D7 → E5 → F7 → H8

✓ A5 → C2:
   Минимум ходов: 3 (ожидалось: 3)
   Путь: A5 → B3 → A1 → C2

✓ A1 → A1:
   Минимум ходов: 0 (ожидалось: 0)
   Путь: A1

✓ A1 → H1:
   Минимум ходов: 5 (ожидалось: 5)
   Путь: A1 → B3 →